# Phase 4: Feature Engineering - The "Vitality Complexity Index"

In this phase, we translate clinical intuition into a programmable risk-scoring algorithm based on the LACE Index to flag high-risk diabetic patients prior to discharge.

In this phase,

1. Define modular scoring functions for Length of Stay (`L`), Acuity (`A`), Comorbidities (`C`), and Emergency Visits (`E`).
2. Programmatically calculate the total `VCI_Score` (`L + A + C + E`) for every patient encounter in `df`.
3. Categorize patients into three risk strata (`Low Risk`, `Medium Risk`, and `High Risk`) based on their `VCI_Score`.
4. Validate the algorithm by visualizing the 30-day readmission rate (`<30`) across each risk stratum.

# Step 4.1: Import Libraries and Load the Enriched Dataset

We import `pandas`, `matplotlib.pyplot`, and `seaborn`, then load the cleaned and enriched dataset (`diabetic_data_enriched.csv`) from the previous phases.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv("data_files/diabetic_data_enriched.csv", low_memory=False)

print("Dataset shape:", df.shape)
display(df[['time_in_hospital', 'admission_type_id', 'number_diagnoses', 'number_emergency', 'readmitted']].head())

Dataset shape: (100114, 50)


,time_in_hospital,admission_type_id,number_diagnoses,number_emergency,readmitted
0,1,6,1,0,NO
1,3,1,9,0,>30
2,2,1,6,0,NO
3,2,1,7,0,NO
4,1,1,5,0,NO


# Step 4.2: Define VCI Component Scoring Functions

Instead of iterating row-by-row with `iterrows()`, we define four modular Python functions corresponding to the LACE scoring rules (`L`, `A`, `C`, and `E`) to apply efficiently across the DataFrame columns.

In [3]:
def calculate_L_points(days):
    if days < 1:
        return 0
    elif 1 <= days <= 4:
        return 1
    elif 5 <= days <= 13:
        return 4
    else:
        return 7


def calculate_A_points(adm_id):
    if int(adm_id) in [1, 7]:
        return 3
    else:
        return 0


def calculate_C_points(diag_count):
    if diag_count < 4:
        return 0
    elif 4 <= diag_count <= 7:
        return 3
    else:  
        return 5


def calculate_E_points(emerg_visits):
    if emerg_visits == 0:
        return 0
    elif 1 <= emerg_visits <= 4:
        return 3
    else: 
        return 5


# Step 4.3: Calculate `VCI_Score` for Every Patient

Apply the four component functions using `.apply()` and sum them (`VCI_Score = L + A + C + E`) to create the `VCI_Score` column.

In [6]:
L = df['time_in_hospital'].apply(calculate_L_points)
A = df['admission_type_id'].apply(calculate_A_points)
C = df['number_diagnoses'].apply(calculate_C_points)
E = df['number_emergency'].apply(calculate_E_points)


df['VCI_Score'] = L + A + C + E

display(df[['time_in_hospital', 'admission_type_id', 'number_diagnoses', 'number_emergency', 'VCI_Score']].head(10))

,time_in_hospital,admission_type_id,number_diagnoses,number_emergency,VCI_Score
0,1,6,1,0,1
1,3,1,9,0,9
2,2,1,6,0,7
3,2,1,7,0,7
4,1,1,5,0,7
5,3,2,9,0,6
6,4,3,7,0,4
7,5,1,8,0,12
8,13,2,8,0,9
9,12,3,8,0,9
